In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

spark = (
    SparkSession.builder
    .appName("HW4.8 Broadcast Join")
    .getOrCreate()
)

orders = spark.read.csv(
    "orders.csv",
    header=True,
    inferSchema=True
)

customers = spark.createDataFrame(
    [
        (10, "premium"),
        (11, "standard"),
        (12, "standard"),
        (13, "premium"),
    ],
    ["customer_id", "segment"]
)

orders = orders.withColumn(
    "revenue",
    F.col("quantity") * F.col("price_each")
)

joined = orders.join(
    broadcast(customers),
    on="customer_id",
    how="inner"
)

result = (
    joined.filter(F.col("status") == "paid")
    .groupBy("segment")
    .agg(
        F.sum("revenue").alias("paid_revenue")
    )
    .orderBy("segment")
)

result.show()

joined.explain()